In [ ]:
import numpy as np
import pandas as pd
# DATA SET
cols = ["age","workclass","fnlwgt","education","education-num",
        "marital-status","occupation","relationship","race","sex",
        "capital-gain","capital-loss","hours-per-week","native-country","income"]
data = pd.read_csv("adult.data", names=cols, na_values=" ?", skipinitialspace=True)
#Remove missing values
data = data.dropna()

data["income"] = data["income"].apply(lambda x: 1 if x == ">50K" else 0)

data = pd.get_dummies(data)      #convert text column into numbers
data = data.astype(float)
X = data.drop("income", axis=1).values
Y = data["income"].values.reshape(1, -1)
X = (X - X.mean()) / X.std()
X = X.T
print("data ready:", X.shape)

# activation functions
def relu(x):
    return np.maximum(0, x)
def drelu(x):
    return x > 0
def sigmoid(x):
    return 1/(1+np.exp(-x))
    
#initialize weights
np.random.seed(1)
w1 = np.random.randn(64, X.shape[0]) * np.sqrt(2/X.shape[0])
b1 = np.zeros((64,1))
w2 = np.random.randn(32, 64) * np.sqrt(2/64)
b2 = np.zeros((32,1))
w3 = np.random.randn(1, 32) * np.sqrt(2/32)
b3 = np.zeros((1,1))

#traning
lr = 0.01
for i in range(1000):
 #forward propagations   
    z1 = np.dot(w1, X) + b1
    a1 = relu(z1)
    z2 = np.dot(w2, a1) + b2
    a2 = relu(z2)
    z3 = np.dot(w3, a2) + b3
    a3 = sigmoid(z3)
    
    m = Y.shape[1]
    loss = -np.sum(Y*np.log(a3 + 1e-8) + (1-Y)*np.log(1-a3 + 1e-8)) / m
    
#backpropagations
    dz3 = a3 - Y
    dw3 = np.dot(dz3, a2.T) / m
    db3 = np.sum(dz3, axis=1, keepdims=True) / m
    da2 = np.dot(w3.T, dz3)
    dz2 = da2 * drelu(z2)
    dw2 = np.dot(dz2, a1.T) / m
    db2 = np.sum(dz2, axis=1, keepdims=True) / m
    da1 = np.dot(w2.T, dz2)
    dz1 = da1 * drelu(z1)
    dw1 = np.dot(dz1, X.T) / m
    db1 = np.sum(dz1, axis=1, keepdims=True) / m
    
 #update weights       
    w1 -= lr * dw1
    b1 -= lr * db1
    w2 -= lr * dw2
    b2 -= lr * db2
    w3 -= lr * dw3
    b3 -= lr * db3
    if i % 100 == 0:
        print("step", i, "loss:", loss)
        
#accuracy
out = sigmoid(np.dot(w3, relu(np.dot(w2, relu(np.dot(w1, X) + b1)) + b2)) + b3)
pred = (out > 0.5).astype(int)
acc = np.mean(pred == Y) * 100
print("accuracy:", acc, "%")

data ready: (108, 32561)
step 0 loss: 1.178409912181775
step 100 loss: 0.562465170437401
step 200 loss: 0.5566932009186709
step 300 loss: 0.5516697240304302
